# PGABL - Tahap 1a: SFT Dataset Preparation

**Proyek**: Fine-tuned Chatbot Tim Legal berbasis RAG
**Peserta**: Nazhif Setya Nugroho
**Tujuan**: Siapkan dataset SFT dari `Ichsan2895/alpaca-gpt4-indonesian` untuk fine-tuning Llama-3.2-3B-Instruct di Tahap 2.

**Design decision penting**: SFT ini mengajarkan model **format bahasa Indonesia formal + instruction-following**, BUKAN pengetahuan hukum. Legal knowledge datang dari RAG (Tahap 3) via retrieval 4 PDF. Ini kenapa dataset generik Alpaca-Indonesian OK — kita bukan menghafal isi PDF ke model.

## Prerequisites

- Colab Runtime: **T4 GPU** (16 GB VRAM). Tokenizer load ringan, GPU tidak wajib tapi untuk konsistensi env.
- Colab Secrets: `HF_TOKEN` (sudah setup di Tahap 0).
- Google Drive mounted: PGABL folder skeleton sudah ada dari Tahap 0.
- Package terinstall dari Tahap 0: `unsloth`, `transformers`, `datasets`.

## Output

- `/content/drive/MyDrive/PGABL/data/processed/sft/train.jsonl`
- `/content/drive/MyDrive/PGABL/data/processed/sft/val.jsonl`
- Setiap baris JSONL: `{"text": "<|begin_of_text|>..."}` — sudah applied Llama-3 chat template.

## Insight dataset

- Dataset punya **2 kolom** (`input`, `output`), BUKAN 3 kolom Alpaca klasik (`instruction`/`input`/`output`).
- Total ~49,969 rows.
- License: CC-BY-SA-4.0.
- `input` = prompt gabungan instruction+context (max ~4.5k char).
- `output` = jawaban (max ~6k char).

## Cell 1: Env detect + load secrets + mount Drive

In [ ]:
import os, sys, json, random
from pathlib import Path

assert 'google.colab' in sys.modules, "Notebook ini butuh Colab (butuh HF_TOKEN via userdata + Drive mount)"

from google.colab import userdata, drive

HF_TOKEN = userdata.get('HF_TOKEN')
assert HF_TOKEN, "HF_TOKEN tidak ditemukan di Colab Secrets"
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGINGFACE_TOKEN'] = HF_TOKEN

# Mount Drive
drive.mount('/content/drive')

# Paths
DRIVE_ROOT = Path('/content/drive/MyDrive/PGABL')
SFT_OUT_DIR = DRIVE_ROOT / 'data' / 'processed' / 'sft'
SFT_OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'HF_TOKEN loaded (starts with: {HF_TOKEN[:6]}...)')
print(f'SFT output dir: {SFT_OUT_DIR}')
print(f'Exists: {SFT_OUT_DIR.exists()}')

## Cell 2: Load dataset dari HuggingFace

Dataset: `Ichsan2895/alpaca-gpt4-indonesian` — instruction-tuning bahasa Indonesia format Alpaca (tapi 2 kolom `input`/`output`, bukan 3 kolom klasik).

**Rubric-locked**: dataset ini WAJIB, tidak boleh diganti.

In [ ]:
from datasets import load_dataset

print('Loading dataset...')
ds = load_dataset('Ichsan2895/alpaca-gpt4-indonesian', split='train')
print(f'Loaded: {len(ds):,} rows')
print(f'Columns: {ds.column_names}')
print(f'Features: {ds.features}')

## Cell 3: EDA - sample + distribusi panjang

Verifikasi struktur dataset + cek distribusi panjang untuk stratified split.

In [ ]:
# 3 sample acak
random.seed(42)
sample_indices = random.sample(range(len(ds)), 3)
for i, idx in enumerate(sample_indices, 1):
    ex = ds[idx]
    print(f'--- Sample {i} (idx {idx}) ---')
    print(f'INPUT ({len(ex["input"])} char):')
    print(ex['input'][:200])
    if len(ex['input']) > 200:
        print('  ...')
    print(f'\nOUTPUT ({len(ex["output"])} char):')
    print(ex['output'][:200])
    if len(ex['output']) > 200:
        print('  ...')
    print()

In [ ]:
# Distribusi panjang
import statistics

input_lens = [len(ex['input']) for ex in ds]
output_lens = [len(ex['output']) for ex in ds]

def stats(name, arr):
    print(f'{name}:')
    print(f'  min: {min(arr)}, max: {max(arr):,}')
    print(f'  mean: {statistics.mean(arr):.1f}, median: {statistics.median(arr):.0f}')
    print(f'  stdev: {statistics.stdev(arr):.1f}')

stats('INPUT length (char)', input_lens)
print()
stats('OUTPUT length (char)', output_lens)

# Histogram bucket count (untuk stratified split)
BUCKET_BOUNDARIES = [100, 500, 1500]  # short / medium / long / very_long
def bucket(l):
    if l < BUCKET_BOUNDARIES[0]: return 'short'
    if l < BUCKET_BOUNDARIES[1]: return 'medium'
    if l < BUCKET_BOUNDARIES[2]: return 'long'
    return 'very_long'

buckets_input = [bucket(l) for l in input_lens]
print(f'\nINPUT length bucket distribution:')
from collections import Counter
for b, count in sorted(Counter(buckets_input).items()):
    print(f'  {b:12s}: {count:>6,}  ({count/len(ds)*100:.1f}%)')

## Cell 4: Filter outlier + split 90/10 stratified

**Filter**:
- `input < 5 char` → skip (kemungkinan garbage)
- `output < 20 char` → skip (jawaban terlalu pendek)

**Split** 90/10 **stratified** by input length bucket (short/medium/long/very_long) dgn seed=42.

**Kenapa stratified**: distribusi panjang harus konsisten antara train & val supaya eval_loss meaningful. Kalau kebetulan val set dominan panjang, model kelihatan buruk padahal train baik.

In [ ]:
# 1. Filter outlier
MIN_INPUT_CHARS = 5
MIN_OUTPUT_CHARS = 20

def keep_example(ex):
    return len(ex['input']) >= MIN_INPUT_CHARS and len(ex['output']) >= MIN_OUTPUT_CHARS

ds_filtered = ds.filter(keep_example)
print(f'Original: {len(ds):,} rows')
print(f'Filtered: {len(ds_filtered):,} rows (dropped {len(ds) - len(ds_filtered):,})')

# 2. Stratified split 90/10 by input length bucket
SEED = 42
TRAIN_RATIO = 0.9

def bucket_id(ex):
    l = len(ex['input'])
    if l < BUCKET_BOUNDARIES[0]: return 0
    if l < BUCKET_BOUNDARIES[1]: return 1
    if l < BUCKET_BOUNDARIES[2]: return 2
    return 3

ds_filtered = ds_filtered.add_column('_bucket', [bucket_id(ex) for ex in ds_filtered])

# Split per bucket
train_indices, val_indices = [], []
for b in [0, 1, 2, 3]:
    bucket_indices = [i for i, ex in enumerate(ds_filtered) if ex['_bucket'] == b]
    random.seed(SEED + b)
    random.shuffle(bucket_indices)
    n_train = int(len(bucket_indices) * TRAIN_RATIO)
    train_indices.extend(bucket_indices[:n_train])
    val_indices.extend(bucket_indices[n_train:])

# Shuffle final untuk mix bucket
random.seed(SEED)
random.shuffle(train_indices)
random.shuffle(val_indices)

train_ds = ds_filtered.select(train_indices)
val_ds = ds_filtered.select(val_indices)

print(f'\nSplit result (stratified by input length):')
print(f'  train: {len(train_ds):,} ({len(train_ds)/len(ds_filtered)*100:.1f}%)')
print(f'  val:   {len(val_ds):,} ({len(val_ds)/len(ds_filtered)*100:.1f}%)')

# Verify stratification
def bucket_dist(sub_ds):
    return {b: sum(1 for ex in sub_ds if ex['_bucket'] == b) for b in [0,1,2,3]}

print(f'\nBucket distribution:')
print(f'  train: {bucket_dist(train_ds)}')
print(f'  val:   {bucket_dist(val_ds)}')

## Cell 5: Load tokenizer Llama-3.2-3B-Instruct + apply chat template

Rubric WAJIB:
1. Pakai chat template Llama-3 via `datasets.map()`.
2. Print output ter-format dgn special tokens (bukti chat template dipakai).

Special tokens Llama-3 yang HARUS muncul:
- `<|begin_of_text|>` di awal
- `<|start_header_id|>{role}<|end_header_id|>` (system/user/assistant)
- `<|eot_id|>` di akhir tiap turn

In [ ]:
from transformers import AutoTokenizer

MODEL_ID = 'unsloth/Llama-3.2-3B-Instruct'
print(f'Loading tokenizer: {MODEL_ID}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
print(f'Tokenizer loaded. vocab_size={tokenizer.vocab_size:,}')
print(f'Chat template: {tokenizer.chat_template[:200] if tokenizer.chat_template else "(default)"}...')

In [ ]:
# Chat template mapping - dataset punya 2 kolom (input, output), bukan 3 kolom klasik
SYSTEM_PROMPT = 'Anda adalah asisten AI yang membantu.'

def format_prompt(example):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': example['input']},
        {'role': 'assistant', 'content': example['output']},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {'text': text}

# Apply chat template via datasets.map() - rubric WAJIB
print('Applying chat template to train...')
train_ds_formatted = train_ds.map(format_prompt, remove_columns=['input', 'output', '_bucket'])
print('Applying chat template to val...')
val_ds_formatted = val_ds.map(format_prompt, remove_columns=['input', 'output', '_bucket'])

print(f'\nFormatted:')
print(f'  train: {len(train_ds_formatted):,}')
print(f'  val:   {len(val_ds_formatted):,}')

## Cell 6: Verify output ter-format (special tokens present)

Rubric WAJIB **print output ter-format** — bukti bahwa chat template Llama-3 sudah diaplikasikan.

In [ ]:
# Print 1 sample ter-format
print('=' * 70)
print('SAMPLE OUTPUT TER-FORMAT (Llama-3 chat template)')
print('=' * 70)
sample_text = train_ds_formatted[0]['text']
print(sample_text)
print('=' * 70)

# Verify special tokens
REQUIRED_TOKENS = [
    '<|begin_of_text|>',
    '<|start_header_id|>system<|end_header_id|>',
    '<|start_header_id|>user<|end_header_id|>',
    '<|start_header_id|>assistant<|end_header_id|>',
    '<|eot_id|>',
]
print('\nVerify special tokens:')
all_ok = True
for tok in REQUIRED_TOKENS:
    present = tok in sample_text
    status = 'OK  ' if present else 'MISS'
    print(f'  [{status}] {tok}')
    if not present:
        all_ok = False

assert all_ok, 'Ada special token yang hilang - chat template mapping gagal'
print('\n=== All special tokens present. Chat template applied correctly. ===')

## Cell 7: Save train.jsonl + val.jsonl ke Drive

Format JSONL: satu baris per example, key `text` berisi full formatted prompt.

In [ ]:
def save_jsonl(dataset, path):
    with open(path, 'w', encoding='utf-8') as f:
        for ex in dataset:
            f.write(json.dumps({'text': ex['text']}, ensure_ascii=False) + '\n')

train_path = SFT_OUT_DIR / 'train.jsonl'
val_path = SFT_OUT_DIR / 'val.jsonl'

print(f'Saving train.jsonl ({len(train_ds_formatted):,} rows)...')
save_jsonl(train_ds_formatted, train_path)
print(f'  Saved: {train_path} ({train_path.stat().st_size / 1024 / 1024:.1f} MB)')

print(f'\nSaving val.jsonl ({len(val_ds_formatted):,} rows)...')
save_jsonl(val_ds_formatted, val_path)
print(f'  Saved: {val_path} ({val_path.stat().st_size / 1024 / 1024:.1f} MB)')

# Verify: baca ulang 1 baris
print('\n--- Verify: read back first line of train.jsonl ---')
with open(train_path, encoding='utf-8') as f:
    first = json.loads(f.readline())
print(f'Text length: {len(first["text"])} char')
print(f'Contains <|begin_of_text|>: {"<|begin_of_text|>" in first["text"]}')
print(f'Contains <|eot_id|>: {"<|eot_id|>" in first["text"]}')

## Cell 8: Save summary

Log dataset stats untuk audit trail.

In [ ]:
summary = {
    'dataset_id': 'Ichsan2895/alpaca-gpt4-indonesian',
    'dataset_columns': ['input', 'output'],
    'total_rows_original': len(ds),
    'total_rows_filtered': len(ds_filtered),
    'rows_dropped': len(ds) - len(ds_filtered),
    'filter_criteria': {
        'min_input_chars': MIN_INPUT_CHARS,
        'min_output_chars': MIN_OUTPUT_CHARS,
    },
    'split_ratio': TRAIN_RATIO,
    'split_strategy': 'stratified by input length bucket',
    'bucket_boundaries': BUCKET_BOUNDARIES,
    'seed': SEED,
    'train_count': len(train_ds_formatted),
    'val_count': len(val_ds_formatted),
    'chat_template': 'llama-3',
    'tokenizer': MODEL_ID,
    'system_prompt': SYSTEM_PROMPT,
    'output_train_path': str(train_path),
    'output_val_path': str(val_path),
}

summary_path = SFT_OUT_DIR / '_summary.json'
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(f'Summary saved: {summary_path}')
print(json.dumps(summary, indent=2))

## ✅ Tahap 1a selesai

Kalau semua cell hijau + verifikasi special tokens present, Tahap 1a ✅ DONE.

**Output di Drive** (`/content/drive/MyDrive/PGABL/data/processed/sft/`):
- `train.jsonl` — training set (90%)
- `val.jsonl` — validation set (10%)
- `_summary.json` — audit trail

**Next**: Tahap 1c (test-set kurasi 30-50 Q&A dari 4 PDF) → Tahap 2 (Fine-tuning SFT + GRPO).